In [1]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.io import savemat

from sciopy import EIT_16_32_64_128, EitMeasurementSetup

In [2]:
# ============================================================
# DEVICE CONNECTION
# ============================================================

n_el = 32
sciospec = EIT_16_32_64_128(n_el)

sciospec.connect_device_HS()

# Optional: clear/check message buffer
sciospec.SystemMessageCallback()

# Optional: get device info
sciospec.GetDeviceInfo()

No message inside the message buffer
message buffer:
 []
message length:	 0
Command-Acknowledge: Command has been executed successfully
message buffer:
 ['0xd1', '0x14', '0x1', '0x0', '0x19', '0x2', '0x65', '0x10', '0x3', '0x3', '0x0', '0x21', '0x0', '0x8d', '0x0', '0xba', '0x0', '0xe4', '0x0', '0xba', '0x0', '0xe4', '0xd1', '0x18', '0x1', '0x83', '0x18']
message length:	 27


In [3]:
# ============================================================
# GENERAL MEASUREMENT SETTINGS
# ============================================================

burst_count = 50
exc_freq = 1_000
framerate = 10
amplitude = 0.01
gain = 1
adc_range = 1

save_dir = Path("triangle_ref_measurements")
save_dir.mkdir(exist_ok=True)

In [4]:
# ============================================================
# AUTOMATED MEASUREMENT LOOP
# ============================================================

for inj_skip in range(17):  # 0, 1, 2, ..., 16

    print(f"\n==============================")
    print(f"Starting measurement for Inj_skip = {inj_skip}")
    print(f"==============================")

    # Create setup for current skip value
    setup = EitMeasurementSetup(
        burst_count=burst_count,
        n_el=n_el,
        exc_freq=exc_freq,
        framerate=framerate,
        amplitude=amplitude,
        inj_skip=inj_skip,
        gain=gain,
        adc_range=adc_range,
    )

    # Send setup to device
    sciospec.SetMeasurementSetup(setup)

    # Small pause to let the device apply settings
    time.sleep(0.5)

    # Optional: ask device to return current setup
    # Useful for debugging, but can be commented out later
    sciospec.GetMeasurementSetup(2)

    # Start measurement
    data = sciospec.StartStopMeasurement(return_as="pot_mat")

    # Check data shape
    assert data.shape[0] == setup.burst_count, (
        f"Wrong burst count for inj_skip={inj_skip}: "
        f"expected {setup.burst_count}, got {data.shape[0]}"
    )

    # Save file
    filename = save_dir / f"triangle_ref_skip{inj_skip:02d}.mat"

    savemat(
        filename,
        {
            "data": data,
            "amplitude": setup.amplitude,
            "n_frames": setup.burst_count,
            "numOfChannels": n_el,
            "NSkip": setup.inj_skip,
            "exc_freq": setup.exc_freq,
            "framerate": setup.framerate,
            "gain": setup.gain,
            "adc_range": setup.adc_range,
        },
    )

    print(f"Saved: {filename}")

    # Optional pause between measurements
    time.sleep(1.0)


Starting measurement for Inj_skip = 0
Command-Acknowledge: Command has been executed successfully
message buffer:
 ['0x18', '0x1', '0x83', '0x18']
message length:	 4
Command-Acknowledge: Command has been executed successfully
message buffer:
 ['0x18', '0x1', '0x83', '0x18']
message length:	 4
Command-Acknowledge: Command has been executed successfully
message buffer:
 ['0x18', '0x1', '0x83', '0x18']
message length:	 4
Command-Acknowledge: Command has been executed successfully
message buffer:
 ['0x18', '0x1', '0x83', '0x18']
message length:	 4
TBD (to be checked)
Command-Acknowledge: Command has been executed successfully
message buffer:
 ['0xb1', '0x3', '0x2', '0x0', '0x32', '0xb1', '0x18', '0x1', '0x83', '0x18']
message length:	 10
TBD: Translation
Saved: triangle_ref_measurements\triangle_ref_skip00.mat

Starting measurement for Inj_skip = 1
Command-Acknowledge: Command has been executed successfully
message buffer:
 ['0x18', '0x1', '0x83', '0x18']
message length:	 4
Command-Acknow

In [5]:
# ============================================================
# RESET DEVICE SOFTWARE
# ============================================================

print("\nAll measurements finished. Resetting device software...")
sciospec.SoftwareReset()
print("Done.")


All measurements finished. Resetting device software...
Command-Acknowledge: Command has been executed successfully
message buffer:
 ['0x18', '0x1', '0x83', '0x18']
message length:	 4
Done.
